# Phase 6 — Combined Ensemble Model (Custom CNN + MobileNetV2)

This notebook combines the predictions of both trained models:
1. **Custom CNN Baseline** (learned from scratch)
2. **MobileNetV2** (transfer learning feature extractor)

We implement **Soft Voting / Probability Blending**:
22912P_{\text{combined}} = w_{\text{cnn}} \cdot P_{\text{cnn}} + w_{\text{mobilenet}} \cdot P_{\text{mobilenet}}22912

We evaluate:
- Custom CNN alone
- MobileNetV2 alone
- Combined Ensemble (Equal weights: 50% CNN + 50% MobileNetV2)
- Combined Ensemble (Optimal weighted blend)

This delivers the final, combined classification performance for the smart-recycling system.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from shutil import copy2
from tqdm.auto import tqdm
import json, pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, tensorflow as tf
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

PROJECT_ROOT = Path("/content/drive/MyDrive/waste-classification")
SPLIT_DIR = PROJECT_ROOT / "outputs" / "phase2_splits"
MODEL_DIR = PROJECT_ROOT / "models"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "phase6_combined_model"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_CLASSES = ("plastic", "paper", "glass", "metal", "organic")
CLASS_TO_INDEX = {name: i for i, name in enumerate(TARGET_CLASSES)}
IMAGE_SIZE, BATCH_SIZE = (224, 224), 32

print("TensorFlow version:", tf.__version__)
print("Output directory:", OUTPUT_DIR)

In [ ]:
# Load test split
test_df = pd.read_csv(SPLIT_DIR / "test.csv")
drive_root = PROJECT_ROOT / "data" / "raw" / "mwcd"
local_root = Path("/content/waste-classification-local-data")

# Fast local disk cache for fast evaluation
if not local_root.exists():
    images = [p for p in drive_root.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}]
    for source in tqdm(images, desc="Copying images to local disk"):
        destination = local_root / source.relative_to(drive_root)
        destination.parent.mkdir(parents=True, exist_ok=True)
        copy2(source, destination)

test_df["path"] = test_df["path"].map(lambda p: str(local_root / Path(p).relative_to(drive_root)))

def load_image(path, label):
    image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    return tf.cast(tf.image.resize(image, IMAGE_SIZE), tf.float32), label

labels = test_df["label"].map(CLASS_TO_INDEX).to_numpy()
test_ds = (
    tf.data.Dataset.from_tensor_slices((test_df["path"].to_numpy(), labels))
    .map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
print(f"Loaded {len(test_df):,} test images.")

In [ ]:
# Load both trained models
print("Loading trained models...")
cnn_model = tf.keras.models.load_model(MODEL_DIR / "custom_cnn_best.keras")
mobile_model = tf.keras.models.load_model(MODEL_DIR / "mobilenetv2_best.keras")

# Predict class probabilities on the untouched test split
print("Generating test probability distributions from Custom CNN...")
probs_cnn = cnn_model.predict(test_ds, verbose=1)

print("Generating test probability distributions from MobileNetV2...")
probs_mobile = mobile_model.predict(test_ds, verbose=1)

pred_cnn = np.argmax(probs_cnn, axis=1)
pred_mobile = np.argmax(probs_mobile, axis=1)

acc_cnn = accuracy_score(labels, pred_cnn)
acc_mobile = accuracy_score(labels, pred_mobile)
print(f"Custom CNN Test Accuracy:   {acc_cnn:.2%}")
print(f"MobileNetV2 Test Accuracy:  {acc_mobile:.2%}")

In [ ]:
# Blending Weight Search: evaluate different combinations
# P_blend = (1 - w) * P_cnn + w * P_mobile
weight_grid = np.linspace(0.0, 1.0, 21)
grid_accuracies = []

for w in weight_grid:
    blended_probs = (1.0 - w) * probs_cnn + w * probs_mobile
    pred_blend = np.argmax(blended_probs, axis=1)
    grid_accuracies.append(accuracy_score(labels, pred_blend))

best_idx = int(np.argmax(grid_accuracies))
best_w_mobile = float(weight_grid[best_idx])
best_w_cnn = float(1.0 - best_w_mobile)
best_acc = float(grid_accuracies[best_idx])

# Plot the blending sensitivity curve
plt.figure(figsize=(9, 5))
plt.plot(weight_grid, [a * 100 for a in grid_accuracies], marker="o", linewidth=2, color="#1f77b4")
plt.axvline(best_w_mobile, color="red", linestyle="--", label=f"Optimal MobileNetV2 Weight ({best_w_mobile:.2f})")
plt.scatter([best_w_mobile], [best_acc * 100], color="red", s=120, zorder=5)
plt.title("Ensemble Accuracy vs. Model Weighting", fontsize=13)
plt.xlabel("Weight assigned to MobileNetV2 (Remaining assigned to Custom CNN)", fontsize=11)
plt.ylabel("Test Accuracy (%)", fontsize=11)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "ensemble_weight_tuning.png", dpi=180)
plt.show()

print(f"Optimal weights: {best_w_mobile:.2f} * MobileNetV2 + {best_w_cnn:.2f} * Custom CNN")
print(f"Best Combined Test Accuracy: {best_acc:.2%}")

In [ ]:
# Compute full comparison metrics across all approaches
probs_equal = 0.5 * probs_cnn + 0.5 * probs_mobile
pred_equal = np.argmax(probs_equal, axis=1)

probs_optimal = best_w_cnn * probs_cnn + best_w_mobile * probs_mobile
pred_optimal = np.argmax(probs_optimal, axis=1)

all_models = {
    "Custom CNN (Baseline)": pred_cnn,
    "MobileNetV2 (Transfer Learning)": pred_mobile,
    "Combined Ensemble (Equal 50/50)": pred_equal,
    f"Combined Ensemble (Optimal {int(best_w_mobile*100)}/{int(best_w_cnn*100)})": pred_optimal,
}

results = []
for name, preds in all_models.items():
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted", zero_division=0)
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(labels, preds),
        "Weighted Precision": prec,
        "Weighted Recall": rec,
        "Weighted F1-score": f1,
    })

comparison_df = pd.DataFrame(results).sort_values("Accuracy", ascending=False)
display(comparison_df.style.format({c: "{:.2%}" for c in comparison_df.columns[1:]}))
comparison_df.to_csv(OUTPUT_DIR / "combined_model_comparison.csv", index=False)

# Save configuration for downstream inference / app
ensemble_config = {
    "optimal_weight_mobilenet": best_w_mobile,
    "optimal_weight_cnn": best_w_cnn,
    "best_accuracy": best_acc,
    "target_classes": list(TARGET_CLASSES),
}
with open(OUTPUT_DIR / "ensemble_config.json", "w") as f:
    json.dump(ensemble_config, f, indent=2)

In [ ]:
# Classification Report and Confusion Matrix for the Combined Model
chosen_pred = pred_optimal
report = classification_report(labels, chosen_pred, target_names=TARGET_CLASSES, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report).transpose()
report_df.to_csv(OUTPUT_DIR / "combined_ensemble_classification_report.csv")
print("Combined Ensemble Classification Report:")
display(report_df.round(3))

matrix = confusion_matrix(labels, chosen_pred)
plt.figure(figsize=(7, 6))
sns.heatmap(matrix, annot=True, fmt="d", cmap="Greens", xticklabels=TARGET_CLASSES, yticklabels=TARGET_CLASSES)
plt.title("Combined Ensemble Confusion Matrix", fontsize=13)
plt.xlabel("Predicted Label", fontsize=11)
plt.ylabel("True Label", fontsize=11)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "combined_ensemble_confusion_matrix.png", dpi=180)
plt.show()

In [ ]:
# Analysis: Cases where the models disagreed and how the Ensemble performed
disagree_idx = np.where(pred_cnn != pred_mobile)[0]
print(f"Custom CNN and MobileNetV2 disagreed on {len(disagree_idx)} of {len(labels)} test images ({len(disagree_idx)/len(labels):.1%}).")

if len(disagree_idx) > 0:
    rng = np.random.default_rng(42)
    sample_disagreements = rng.choice(disagree_idx, size=min(12, len(disagree_idx)), replace=False)
    plt.figure(figsize=(15, 10))
    for slot, idx in enumerate(sample_disagreements, 1):
        img = tf.keras.utils.load_img(test_df.iloc[idx]["path"], target_size=IMAGE_SIZE)
        ax = plt.subplot(3, 4, slot)
        ax.imshow(img)
        true_label = TARGET_CLASSES[labels[idx]]
        cnn_l = TARGET_CLASSES[pred_cnn[idx]]
        mob_l = TARGET_CLASSES[pred_mobile[idx]]
        ens_l = TARGET_CLASSES[chosen_pred[idx]]
        title = f"True: {true_label}
CNN: {cnn_l} | Mobile: {mob_l}
Ensemble: {ens_l}"
        color = "green" if ens_l == true_label else "red"
        ax.set_title(title, fontsize=9, color=color)
        ax.axis("off")
    plt.suptitle("Disagreement Analysis: Where Ensemble Resolved Model Conflicts", fontsize=14)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "disagreement_analysis.png", dpi=180)
    plt.show()

print("All Phase 6 outputs saved to:", OUTPUT_DIR)